In [1]:
# 1. Import Required Libraries (PySpark)
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, split, lower, array_distinct, when, lit, collect_set, array_union
import os

spark = SparkSession.builder \
    .appName("NutritionalValuesGenerator") \
    .config("spark.executor.memory", "4g") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

# In cluster use the HDFS path prefix
# path_prefix = "hdfs:///projects/BDA-12/"
# In local use the local path prefix
path_prefix = "" 

spark.sparkContext.setLogLevel("WARN")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/01/12 17:38:16 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/01/12 17:38:17 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/01/12 17:38:17 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


In [ ]:
base_dir = f"{path_prefix}converted-dataset"

# 2. Load only necessary Parquet Files as Spark DataFrames
food = spark.read.parquet(os.path.join(base_dir, "food.parquet"))
branded_food = spark.read.parquet(os.path.join(base_dir, "branded_food.parquet"))
food_attribute = spark.read.parquet(os.path.join(base_dir, "food_attribute.parquet"))

In [3]:
# 3. Merge food and branded_food for description and ingredients
food_merged = food.join(branded_food, "fdc_id", "left")

In [4]:
# 4. Extract and Normalize Ingredients from branded_food and food_attribute
food_merged = food_merged.withColumn("ingredients_list", split(lower(col("ingredients")), ",|;|CONTAINS:"))

attr_ingredients = food_attribute.filter(col("name") == "Ingredients")
attr_ingredients = attr_ingredients.groupBy("fdc_id").agg(collect_set("value").alias("attr_ingredients_raw"))
attr_ingredients = attr_ingredients.withColumn("attr_ingredients", split(lower(col("attr_ingredients_raw")[0]), ",|;|CONTAINS:"))

food_merged = food_merged.join(attr_ingredients.select("fdc_id", "attr_ingredients"), "fdc_id", "left")

food_merged = food_merged.withColumn(
    "all_ingredients",
    array_distinct(
        array_union(
            when(col("ingredients_list").isNotNull(), col("ingredients_list")).otherwise(array_distinct(lit([]))),
            when(col("attr_ingredients").isNotNull(), col("attr_ingredients")).otherwise(array_distinct(lit([])))
        )
    )
)

In [5]:
# 5. Select only ingredients and product link columns
ingredient_cols = ["fdc_id", "description", "all_ingredients"]
custom_df = food_merged.select(*[col(c) for c in ingredient_cols if c in food_merged.columns])

In [6]:
# 6. Save to Parquet
output_dir = "output/ingredients_nutrional_profiles"
os.makedirs(output_dir, exist_ok=True)
custom_df.write.mode("overwrite").parquet(os.path.join(output_dir, "ingredients_nutrional_profiles.parquet"))
print(f"Saved ingredients nutritional profiles to {output_dir}/ingredients_nutrional_profiles.parquet")

Saved ingredients nutritional profiles to output/ingredients_nutrional_profiles/ingredients_nutrional_profiles.parquet
